In [0]:
from pyspark.sql.functions import *

# Read Bronze TXT Table
txt_df = spark.table("workauto.bronze.txt_raw")

# Remove Duplicate Records
txt_df = txt_df.dropDuplicates()

# Trim Spaces
txt_df = txt_df.withColumn(
    "raw_text",
    trim(col("raw_text"))
)

# Standardize Text
txt_df = txt_df.withColumn(
    "clean_text",
    lower(col("raw_text"))
)

# Remove Very Short Messages
txt_df = txt_df.filter(length(col("clean_text")) > 10)

# Categorize Complaints Automatically
txt_df = txt_df.withColumn(
    "ticket_category",
    when(col("clean_text").contains("refund"), "Refund Request")
    .when(col("clean_text").contains("payment"), "Payment Issue")
    .when(col("clean_text").contains("delivery"), "Delivery Issue")
    .when(col("clean_text").contains("shipment"), "Shipment Delay")
    .otherwise("General Inquiry")
)

# Assign Priority
txt_df = txt_df.withColumn(
    "priority",
    when(col("ticket_category") == "Refund Request", "High")
    .when(col("ticket_category") == "Payment Issue", "High")
    .when(col("ticket_category") == "Shipment Delay", "Medium")
    .otherwise("Low")
)

# Workflow Automation Logic
txt_df = txt_df.withColumn(
    "workflow_status",
    when(col("priority") == "High", "Escalate to Support Team")
    .when(col("priority") == "Medium", "Assign to Logistics Team")
    .otherwise("Auto-Resolved")
)

# Add Processing Timestamp
txt_df = txt_df.withColumn(
    "processed_time",
    current_timestamp()
)

# Show Data
txt_df.show(truncate=False)

# Save Silver Table
(
    txt_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workauto.silver.txt_clean")
)

print("TXT Silver Table Created Successfully")